In [ ]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("../..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

01:26:02 [INFO] Device: cuda


In [2]:
CFG = {
    "task":        "lo",
    "dataset":     "kcnh2",
    "fp_type":     "rdkit_desc",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {"train": train_df, "test": test_df}

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} | "
        f"test={len(test_df)} "
        f"n_clusters={test_df['cluster'].nunique()}"
    )

folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

01:26:02 [INFO] Loaded lo/kcnh2 fold 1: train=3313, test=406
01:26:02 [INFO] Fold 1 | train=3313 y_mean=5.662 y_std=0.561 | test=406 n_clusters=34
01:26:02 [INFO] Loaded lo/kcnh2 fold 2: train=3313, test=406
01:26:02 [INFO] Fold 2 | train=3313 y_mean=5.663 y_std=0.562 | test=406 n_clusters=34
01:26:02 [INFO] Loaded lo/kcnh2 fold 3: train=3313, test=406
01:26:02 [INFO] Fold 3 | train=3313 y_mean=5.662 y_std=0.561 | test=406 n_clusters=34
01:26:02 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_train_1.npz
01:26:02 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_test_1.npz


01:26:02 [INFO] Fold 1 | X_train: (3313, 217), X_test: (406, 217) | scale_features=True | y_mean=5.662 y_std=0.561
01:26:02 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_train_2.npz
01:26:02 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_test_2.npz
01:26:02 [INFO] Fold 2 | X_train: (3313, 217), X_test: (406, 217) | scale_features=True | y_mean=5.663 y_std=0.562
01:26:02 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_train_3.npz
01:26:02 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kcnh2/rdkit_desc_test_3.npz
01:26:02 [INFO] Fold 3 | X_train: (3313, 217), X_test: (406, 217) | scale_features=True | y_mean=5.662 y_std=0.561


In [5]:
# ── Block 4 ── Nested CV random search ─────────────────────────────────────

logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(fold_results, title="MLP KCNH2 Lo — rdkit")

01:26:02 [INFO] Estimated time: ~45 min
01:26:02 [INFO] 
OUTER FOLD 1
01:26:08 [INFO]   [1/150] inner MSE=0.7706 (score=-0.7706) (6s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
01:26:12 [INFO]   [2/150] inner MSE=0.8242 (score=-0.8242) (5s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
01:26:14 [INFO]   [3/150] inner MSE=0.8502 (score=-0.8502) (2s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
01:26:16 [INFO]   [4/150] inner MSE=0.8113 (score=-0.8113) (3s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
01:26:18 [INFO]   [5/150] inner MSE=0.7981 (score=-0.7981) (2s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
01:26:26 [INFO]   [6/150] inner MSE=0.8806 (score=-0.8806) (8s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
01:26:34 [INFO]   [7/150] inner MSE=0.8042 (score=-0.8042) (7s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
01:26:40 [INFO]   [8/150] inner MSE=0.8764 (score=-0.8764) (7s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
01:26:49 [INFO]   [9/150] inner MSE=0.8190 (score=-0.8190) (8s) | L=1 N=256 r=0.5 dropout=0.3 lr=1e-04
01:26


MLP KCNH2 Lo — rdkit
Fold 1: mean_spearman=0.4817
Fold 2: mean_spearman=0.4149
Fold 3: mean_spearman=0.3476

Aggregated metrics:
  mean_spearman_mean: 0.4147
  mean_spearman_std: 0.0547
  std_spearman_mean: 0.4006
  std_spearman_std: 0.0263
  mean_r2_mean: -0.6755
  mean_r2_std: 0.0332
  mean_mae_mean: 0.8855
  mean_mae_std: 0.0028
  n_clusters_mean: 34.0
  n_clusters_std: 0.0

Best hyperparameters per fold:
Fold 1: L=2 N=512 r=0.5 act=leaky_relu dropout=0.3 bn=False init=xavier lr=3e-03 wd=1e-02 bs=256
Fold 2: L=3 N=1024 r=0.5 act=relu dropout=0.3 bn=False init=xavier lr=1e-03 wd=1e-02 bs=128
Fold 3: L=3 N=1024 r=0.7 act=leaky_relu dropout=0.3 bn=False init=xavier lr=3e-03 wd=1e-02 bs=256


{'mean_spearman_mean': np.float64(0.4147),
 'mean_spearman_std': np.float64(0.0547),
 'std_spearman_mean': np.float64(0.4006),
 'std_spearman_std': np.float64(0.0263),
 'mean_r2_mean': np.float64(-0.6755),
 'mean_r2_std': np.float64(0.0332),
 'mean_mae_mean': np.float64(0.8855),
 'mean_mae_std': np.float64(0.0028),
 'n_clusters_mean': np.float64(34.0),
 'n_clusters_std': np.float64(0.0)}

In [6]:
import json
from pathlib import Path

out_dir = PROJECT_ROOT / "results" / CFG["task"] / CFG["dataset"] / f"mlp_{CFG['dataset']}_{CFG['task']}_{CFG['fp_type']}"
out_dir.mkdir(parents=True, exist_ok=True)

for res in fold_results:
    fold = res["fold"]

    with open(out_dir / f"params_fold_{fold}.json", "w") as f:
        json.dump({
            "fold": fold,
            "best_params": res["best_hp"],
            "inner_cv_score": res["inner_score"],
            "inner_train_loss_diagnostic": res["best_train_loss_diagnostic"],
            "test_metrics": res["test_metrics"],
            "seed_results": res["seed_results"],
        }, f, indent=2, default=str)

print(f"Saved JSON files in: {out_dir}")

Saved JSON files in: /home/f.capria/drug-discovery-lohi/results/lo/kcnh2/mlp_kcnh2_lo_rdkit_desc
